# Flow Matching on Trametinib Single-Cell PCA Data

Mirrors the structure of `ChatterjeeLab/CIS6270` lecture_3 `esm2_flow_guidance.py`, but swaps the ESM-2 encoder + reward guidance for this project's own `TrametinibSingleBranchDataModule`.

**Convention:** `Z_0` = noise (by default fit to the real DMSO / untreated population's mean & covariance, not a plain `N(0,I)`), `Z_1` = real Trametinib-treated cells. The model learns a velocity field transporting `Z_0 -> Z_1`.

In [ ]:
import sys
from pathlib import Path
from types import SimpleNamespace

import torch
from torch import nn
import torch.nn.functional as F

ROOT = Path.cwd()  # run this notebook from the project root
sys.path.insert(0, str(ROOT))
from dataloaders.trametinib_loader import TrametinibSingleBranchDataModule

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HIDDEN, LEARNING_RATE = 128, 1e-3

## Config

Notebook equivalent of the script's `argparse` flags -- edit values directly instead of passing CLI args.

In [ ]:
args = SimpleNamespace(
    data_path=ROOT / "data" / "Trametinib_5.0uM_pca_and_leidenumap_labels.csv",
    batch_size=64,
    dim=50,
    whiten=False,
    split_ratios=[0.8, 0.2],
    epochs=200,
    lr=LEARNING_RATE,
    steps=100,
    guidance_eta=0.0,
    noise_source="data",  # "data" fits Z_0 to the real DMSO mean/covariance; "standard" uses N(0,I)
    samples=8,
    seed=7,
    output=ROOT / "flow_outputs",
)

torch.manual_seed(args.seed)
if DEVICE.type == "cpu":
    torch.set_num_threads(2)

## 1. Data loading

The project's own `LightningDataModule` parses the CSV, builds the Trametinib-treated (`X1`) cluster and real DMSO (`X0`) population, and exposes a k-NN tree over the full data for manifold guidance.

In [ ]:
dm = TrametinibSingleBranchDataModule(args)
dim = dm.coords_t1.shape[1]
dim

## Noise source ($Z_0$)

By default this fits a Gaussian to the real DMSO (untreated) cells so sampled noise matches their location/scale/correlations, instead of a generic $N(0,I)$ prior. Set `args.noise_source = "standard"` to fall back to plain $N(0,I)$.

In [ ]:
def fit_noise_source(dm, dim, kind="data"):
    if kind == "standard":
        return {"mean": torch.zeros(1, dim), "chol": torch.eye(dim)}
    if kind != "data":
        raise ValueError("noise-source must be 'data' or 'standard'")
    x0 = dm.coords_t0
    mean = x0.mean(0, keepdim=True)
    cov = torch.cov(x0.T) + 1e-4 * torch.eye(dim)  # Ridge for numerical stability.
    chol = torch.linalg.cholesky(cov)
    return {"mean": mean, "chol": chol}


def sample_noise(stats, n, dim):
    return stats["mean"].to(DEVICE) + torch.randn(n, dim, device=DEVICE) @ stats["chol"].to(DEVICE).T


noise_stats = fit_noise_source(dm, dim, args.noise_source)

## 2. Velocity-field model

Small MLP: flattening lets the output depend on the whole PCA vector at once, same design as the reference `FlowModel`.

In [ ]:
class FlowModel(nn.Module):
    def __init__(self, dim, hidden=HIDDEN):
        super().__init__()
        self.dim = dim
        self.time = nn.Sequential(nn.Linear(1, 32), nn.SiLU(), nn.Linear(32, 32))
        self.skip = nn.Linear(32, 1)  # Time-gated linear passthrough of the state.
        self.net = nn.Sequential(
            nn.Linear(dim + 32, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, dim),
        )

    def forward(self, z, t):
        time = self.time(t[:, None])
        inputs = torch.cat([z, time], dim=1)
        return self.skip(time) * z + self.net(inputs)
        # A time-dependent linear part plus a learned nonlinear velocity correction.

## 3. Flow matching training

$Z_t = (1-t) Z_0 + t Z_1$ with $Z_0 \sim N(\text{noise\_mean}, \text{noise\_cov})$; target velocity $Z_1 - Z_0$.

In [ ]:
def _unwrap_train_batch(batch):
    # dm.train_dataloader() nests two CombinedLoaders; iterating it yields
    # (nested_batch_dict, batch_idx, dataloader_idx) regardless of mode.
    x1, _weights = batch["train_samples"]["x1"]
    return x1


def train(dm, noise_stats, model, epochs, lr=LEARNING_RATE):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loader = dm.train_dataloader()
    for epoch in range(epochs):
        total, n_batches = 0.0, 0
        for batch, _batch_idx, _dataloader_idx in loader:
            z1 = _unwrap_train_batch(batch).to(DEVICE)  # Real Trametinib-treated cells, raw PCA scale.
            z0 = sample_noise(noise_stats, len(z1), model.dim)  # Data-fit (or standard) noise source.
            t = torch.rand(len(z1), device=DEVICE)
            zt = (1 - t[:, None]) * z0 + t[:, None] * z1
            loss = F.mse_loss(model(zt, t), z1 - z0)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            total += loss.item()
            n_batches += 1
        if (epoch + 1) % 50 == 0 or epoch + 1 == epochs:
            print(f"epoch {epoch+1}: flow matching loss {total / max(n_batches, 1):.4f}")
    model.eval().requires_grad_(False)
    for parameter in model.parameters():
        parameter.grad = None
    return model

## 4. Manifold guidance

Pulls the trajectory toward the k-NN-smoothed data manifold the dataloader already builds (`dm.tree` / `dm.dataset`), in place of the reward-gradient guidance the reference script used for sequence objectives.

In [ ]:
def manifold_guidance(dm, z, t, eta):
    if eta == 0:
        return torch.zeros_like(z)
    proj_fn = dm.get_manifold_proj(None)
    projected = proj_fn(z.detach().cpu()).to(z.device)
    kappa = eta * 4 * t[:, None] * (1 - t[:, None])  # Fades guidance at both endpoints.
    return kappa * (projected - z)

## 5. Sampling

Euler-integrate from the fitted noise source to $t=1$, optionally steering with manifold guidance.

In [ ]:
@torch.no_grad()
def sample(model, dm, noise_stats, n, steps=100, eta=0.0):
    z = sample_noise(noise_stats, n, model.dim)  # Z_0, representing the untreated cells.
    dt = 1.0 / steps
    for step in range(steps):
        t = torch.full((n,), step * dt, device=DEVICE)
        v = model(z, t)
        if eta != 0:
            v = v + manifold_guidance(dm, z, t, eta) / dt
        z = z + dt * v
    return z

## 6. Evaluation

Mean distance of generated cells to each metric-sample cluster centroid, mirroring the reference's printed composition proxies.

In [ ]:
@torch.no_grad()
def evaluate(dm, generated):
    generated = generated.cpu()
    for i, loader in enumerate(dm.metric_samples_dataloaders):
        cluster = next(iter(loader))
        centroid = cluster.mean(dim=0)
        dist = (generated - centroid).norm(dim=1).mean().item()
        print(f"  mean distance to metric cluster {i}: {dist:.4f}")

## Run: train the model

In [ ]:
model = FlowModel(dim).to(DEVICE)
train(dm, noise_stats, model, args.epochs, args.lr)

## Run: sample and evaluate

In [ ]:
outputs = {
    "baseline": sample(model, dm, noise_stats, args.samples, steps=args.steps, eta=0.0),
    "manifold_guided": sample(model, dm, noise_stats, args.samples, steps=args.steps, eta=args.guidance_eta),
}
for name, generated in outputs.items():
    if not torch.isfinite(generated).all():
        raise RuntimeError(f"Nonfinite {name} output; reduce guidance_eta or check training")
    print(f"{name}:")
    evaluate(dm, generated)

## Save results

In [ ]:
args.output.mkdir(parents=True, exist_ok=True)
torch.save({
    "model": model.state_dict(),
    "outputs": {k: v.cpu() for k, v in outputs.items()},
    "noise_stats": {k: v.cpu() for k, v in noise_stats.items()},
    "args": vars(args),
}, args.output / "results.pt")
print(f"Saved model weights and generated coordinates to {args.output}")